# MultiQTL tutorial

Step-by-step eQTL / pQTL analysis in Python. Every step can also be done from the browser interface
(`python -m multiqtl gui`) or the command line (`python -m multiqtl run config.yaml`).

This notebook simulates a small study so it runs anywhere in a few minutes. To use your own data, replace the
loading cell with `multiqtl.io.load_dataset(...)` or `pd.read_csv(...)`.

In [ ]:
import numpy as np, pandas as pd
import multiqtl
from multiqtl import preprocess as PP, fdr as F, integration as I, stats as ST, plotting as PL
from multiqtl.mapping import QTLMapper, significant_pairs
from multiqtl.simulate import simulate_study
print("MultiQTL", multiqtl.__version__)

## 1. Data
Simulated: 200 samples, genotypes, RNA-seq counts, LC-MS protein intensities, covariates (age, sex, batch).

In [ ]:
d = simulate_study(None, n_samples=200, n_var_per_chrom=800, n_genes=120, n_proteins=80, seed=1)
for k in ["genotypes", "rna_counts", "protein_intensity", "covariates"]:
    print(f"{k:18s} {d[k].shape}")
d["covariates"].head()

## 2. Quality control and normalization

In [ ]:
G, vqc = PP.genotype_qc(d["genotypes"], maf_min=0.05)
V = d["variants"].loc[G.index]
rna, rna_qc = PP.normalize_expression(d["rna_counts"])            # TMM -> inverse normal
prot, prot_qc = PP.normalize_proteomics(d["protein_intensity"])     # log2 -> filter -> impute -> INT
print(len(G), "variants pass QC;", rna.shape[0], "genes;", prot.shape[0], "proteins")

## 3. Covariates: known + genotype PCs + hidden factors

In [ ]:
known = PP.encode_covariates(d["covariates"])
gpc, _ = PP.pca(G.values, 3)
base = pd.concat([known, pd.DataFrame(gpc, index=G.columns, columns=["gPC1", "gPC2", "gPC3"])], axis=1)
cov_rna = pd.concat([base, PP.hidden_factor_covariates(rna, "auto", known=base).add_prefix("rna_")], axis=1)
cov_prot = pd.concat([base, PP.hidden_factor_covariates(prot, "auto", known=base).add_prefix("prot_")], axis=1)
print(cov_rna.shape[1], "covariates for RNA,", cov_prot.shape[1], "for protein")

## 4. cis-QTL mapping with permutations

In [ ]:
M_e = QTLMapper(G, V, rna, d["gene_annot"].loc[rna.index], cov_rna, name="eQTL")
M_p = QTLMapper(G, V, prot, d["protein_annot"].loc[prot.index], cov_prot, name="pQTL")
nom_e, nom_p = M_e.map_cis_nominal(), M_p.map_cis_nominal()
perm_e = M_e.map_cis_permutation(n_perm=500)
perm_p = M_p.map_cis_permutation(n_perm=500)
print("eGenes:", (perm_e.qval <= 0.05).sum(), "/", len(perm_e), "  pGenes:", (perm_p.qval <= 0.05).sum(), "/", len(perm_p))
perm_p.sort_values("pval_beta").head()[["feature_id", "variant_id", "slope", "pval_nominal", "pval_beta", "qval"]]

## 5. FDR: compare correction methods

In [ ]:
F.fdr_summary(perm_p.pval_beta.values)

## 6. eQTL–pQTL integration: sharing and colocalization

In [ ]:
mp = I.protein_gene_map(d["protein_annot"].loc[prot.index], d["gene_annot"].loc[rna.index])
cmp_ = I.compare_qtls(nom_e, nom_p, mp, perm_e, perm_p)
co = I.coloc_all(cmp_["pairs"])
print(cmp_["leads"].category.value_counts().to_dict())
co.sort_values("PP.H4", ascending=False).head()[["protein_id", "PP.H3", "PP.H4", "top_shared_variant"]]

## 7. Mediation: does the pQTL act through mRNA?

In [ ]:
med = I.mediation_all(cmp_["leads"], G, rna, prot, base, n_boot=300)
med[["protein_id", "indirect", "indirect_ci_low", "indirect_ci_high", "prop_mediated", "mechanism"]].head()

## 8. Figures (shown here; `PL.save_figure(fig, 'name', dpi=600)` writes PNG/TIFF/PDF/SVG)

In [ ]:
%matplotlib inline
PL.qq_plot({"eQTL": nom_e.pval.values, "pQTL": nom_p.pval.values}, "cis nominal P values")

In [ ]:
PL.effect_concordance(cmp_["leads"])

## 9. Power for a follow-up study

In [ ]:
ST.power_grid(n_values=(200, 500), alpha=1e-5).pivot_table(index=["n", "maf"], columns="beta", values="power").round(2)